In [16]:
import os
import re
import warnings

from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows",100)
pd.set_option("display.width",None)

In [17]:
PROJECT_DIR = Path.cwd().parent

DATA_DIR = PROJECT_DIR / "Extracted_Data"

OUTPUT_DIR = PROJECT_DIR / "processed"

OUTPUT_DIR.mkdir(exist_ok=True)

print(DATA_DIR)

c:\Users\vikhy\Desktop\Nikitha project\Extracted_Data


In [18]:
excel_files = sorted(DATA_DIR.rglob("*.xlsx"))

print("="*60)

print("Total Excel Files :",len(excel_files))

print("="*60)

excel_files[:10]

Total Excel Files : 24


[WindowsPath('c:/Users/vikhy/Desktop/Nikitha project/Extracted_Data/DZwEI 08.10-13.10/DZwEI 08.10-13.10/Glauburgstraße 50.1270654,8.6893089/2024-10-08_to_2024-10-13_UI_202403B0923.xlsx'),
 WindowsPath('c:/Users/vikhy/Desktop/Nikitha project/Extracted_Data/DZwEI 08.10-13.10/DZwEI 08.10-13.10/Hanauer Landstraße 50.1125629,8.7039647/2024-10-08_to_2024-10-13_UI_202310B0614.xlsx'),
 WindowsPath('c:/Users/vikhy/Desktop/Nikitha project/Extracted_Data/DZwEI 08.10-13.10/DZwEI 08.10-13.10/Hanauer Landstraße 50.1134021,8.6969654/2024-10-08_to_2024-10-13_UI_202403B0924.xlsx'),
 WindowsPath('c:/Users/vikhy/Desktop/Nikitha project/Extracted_Data/DZwEI 08.10-13.10/DZwEI 08.10-13.10/Töngesgasse 50.1135489,8.6857431/2024-10-08_to_2024-10-13_UI_202403B0908.xlsx'),
 WindowsPath('c:/Users/vikhy/Desktop/Nikitha project/Extracted_Data/DZwEI 09.09-15.09/DZwEI 09.09-15.09/B44/2024-09-09_to_2024-09-15_UI_202403B0908.xlsx'),
 WindowsPath('c:/Users/vikhy/Desktop/Nikitha project/Extracted_Data/DZwEI 09.09-15.09/D

In [19]:
def extract_metadata(filepath):

    week = filepath.parents[1].name

    site = filepath.parent.name

    latitude = np.nan
    longitude = np.nan

    pattern = r'([-+]?\d+\.\d+),([-+]?\d+\.\d+)'

    match = re.search(pattern,site)

    if match:

        latitude=float(match.group(1))

        longitude=float(match.group(2))

        site = re.sub(pattern,'',site).strip()

    return{

        "Week":week,

        "Measurement_Site":site,

        "Latitude":latitude,

        "Longitude":longitude,

        "Source_File":filepath.name

    }

In [25]:
import pandas as pd

def detect_header(file, sheet):
    """
    Detect the header row by searching for keywords like 'Uhrzeit', 'Zeit',
    'Intervall', or 'Time'.
    """

    temp = pd.read_excel(
        file,
        sheet_name=sheet,
        header=None
    )

    keywords = [
        "Uhrzeit",
        "Zeit",
        "Intervall",
        "Time",
        "Von",
        "Nach"
    ]

    # Search first 40 rows
    for row in range(min(40, len(temp))):

        # Convert every cell safely to string
        values = temp.iloc[row].fillna("").astype(str).tolist()

        text = " ".join(map(str, values))

        for key in keywords:
            if key.lower() in text.lower():
                return row

    return None

In [21]:
IGNORE={

    "Erklärungen",

    "Explanation",

    "Info"

}

In [22]:
sample=excel_files[0]

excel=pd.ExcelFile(sample)

excel.sheet_names

['Gesamt-Kfz',
 'SV >3.5t',
 'LV <3.5t',
 'Pkw',
 'Pkw mit Anhänger',
 'Lkw',
 'Lkw mit Anhänger',
 'Bus',
 'Lieferwagen',
 'Kraftrad',
 'Sattel-Kfz',
 'Erklärungen']

In [26]:
sample = excel_files[0]

excel = pd.ExcelFile(sample)

for sheet in excel.sheet_names:

    if sheet in IGNORE:
        continue

    header = detect_header(sample, sheet)

    print(f"{sheet:25} -> {header}")

Gesamt-Kfz                -> 2
SV >3.5t                  -> 2
LV <3.5t                  -> 2
Pkw                       -> 2
Pkw mit Anhänger          -> 2
Lkw                       -> 2
Lkw mit Anhänger          -> 2
Bus                       -> 2
Lieferwagen               -> 2
Kraftrad                  -> 2
Sattel-Kfz                -> 2


In [27]:
header = detect_header(sample, "Gesamt-Kfz")

df = pd.read_excel(
    sample,
    sheet_name="Gesamt-Kfz",
    header=header
)

print(df.head())
print(df.columns.tolist())

    Zähldatum Unnamed: 1                                  Tue, 08. Oct 2024  \
0   Startzeit        NaN                                              00:00   
1    Standort        NaN                                     (50.127,8.689)   
2  Gesamt-Kfz        NaN  Pkw, Pkw mit Anhänger, Kraftrad, Lieferwagen, ...   
3         NaN        NaN                                                NaN   
4         NaN        NaN                                                NaN   

  Unnamed: 3 Unnamed: 4  Unnamed: 5  Unnamed: 6 Unnamed: 7 Unnamed: 8  \
0        NaN        NaN         NaN         NaN        NaN        NaN   
1        NaN        NaN         NaN         NaN        NaN        NaN   
2        NaN        NaN         NaN         NaN        NaN        NaN   
3        NaN        NaN         NaN         NaN        NaN        NaN   
4        NaN        NaN         NaN         NaN        NaN        NaN   

   Unnamed: 9  Unnamed: 10  Unnamed: 11  Unnamed: 12  Unnamed: 13  \
0         NaN    